# Building LLM Applications: Agents, Memory & RAG
### Complete Course Notebook

---

## 📋 Table of Contents
- **Day 1**
  - [Session 1: Prompt Reliability Patterns](#session-1)
  - [Session 2: Local vs Cloud Models + Ollama Setup](#session-2)
  - [Session 3: Tool Calling + Agent with FastAPI](#session-3)
- **Day 2**
  - [Session 4: RAG with LangChain + ChromaDB](#session-4)
  - [Session 5: Orchestration Patterns](#session-5)
  - [Session 6: Tracing + Evals with LangSmith](#session-6)

## ⚙️ Prerequisites & Installation

Run this cell first to install all required packages.

In [ ]:
# In terminal
# python -m venv venv
# venv\Scripts\activate 

# pip install ipykernel
# python -m ipykernel install --user --name=venv --display-name "Python (venv)"

# In your notebook: Kernel → Change Kernel → Python (venv)

In [1]:
# Install all dependencies
!pip install fastapi uvicorn langchain langchain-ollama langchain-community \
             langchain-text-splitters langchainhub python-dotenv \
             chromadb pypdf langsmith pydantic

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp314-cp314-win_amd64.whl.metadata (7.4 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.wh

Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
!pip install langgraph

Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 🦙 Ollama Setup (run these in your Terminal, not here)

```bash
# 1. Download Ollama from https://ollama.com (macOS, Windows, Linux)

# 2. Verify installation
ollama --version

# 3. Pull models
ollama pull qwen3:8b
ollama pull nomic-embed-text   # for RAG embeddings

# 4. Start the Ollama server
ollama serve
```

---
# 📅 DAY 1

## Session 1: Prompt Reliability Patterns

### Key Concepts
| Term | Description |
|------|-------------|
| **Context Window** | Max tokens the model can "see" (Claude Sonnet ~200k, GPT-4o ~128k) |
| **Tokens** | Units of text the LLM processes (words/subwords) |
| **Temperature** | Controls randomness: 0 = deterministic, >1.5 = chaotic |
| **Latency** | How long you wait — longer output = longer wait |
| **Cost** | Priced per 1M tokens in + out |

### The 4 Prompt Reliability Patterns

In [2]:
# ── Pattern 1: Give it a Job Title, Not Just a Task ──────────────────────────

bad_prompt_1 = "Answer the user's coding questions."

good_prompt_1 = """
You are a senior backend engineer specialised in Python and PostgreSQL,
helping a team of mid-level developers during a code review session.
"""

# ── Pattern 2: Rules Are Better Than Vibes ───────────────────────────────────

bad_prompt_2 = "Be professional and don't answer irrelevant questions."

good_prompt_2 = """
RULES
1. Only answer questions about Python, SQL, or system design.
2. If the user asks about anything else, say exactly: "That's outside my scope here."
3. Always show code before explaining it.
4. Maximum response length: 300 words.
5. Never say "As an AI language model."
"""

# ── Pattern 3: Show It, Don't Just Tell It (Few-shot Examples) ───────────────

good_prompt_3 = """
EXAMPLE OF A GOOD RESPONSE
User: "What's the difference between a list and a tuple?"
Answer: "Lists are mutable — you can change them after creation.
Tuples are immutable. Use tuples for fixed data, lists when contents will change."

EXAMPLE OF A BAD RESPONSE
User: "What's the difference between a list and a tuple?"
Answer: "Great question! Both lists and tuples are incredibly useful data structures
in Python with some key similarities but also some important differences worth understanding..."
"""

# ── Pattern 4: Tell It What To Do When It's Stuck ────────────────────────────

good_prompt_4 = """
WHEN YOU DON'T KNOW
If you don't have enough information to answer accurately, say:
"I need more context to answer this properly. Could you share [specific thing]?"
Never guess at specifics like variable names, function signatures,
or database schemas you haven't been shown.

WHEN THE QUESTION IS OUT OF SCOPE
Say exactly: "That's outside what I can help with here."
Do not apologise more than once.
Do not offer alternatives outside the defined scope.
"""

print("✅ Prompt patterns defined.")

✅ Prompt patterns defined.


In [3]:
# ── Structured Output with Schemas ───────────────────────────────────────────
# Instead of free text, ask the model to fill in a form (JSON schema).

structured_output_prompt = """
Analyze the following code and return a JSON object with ONLY these fields:
{
  "issues": ["list of issues found"],
  "severity": "low | medium | high | critical",
  "summary": "one-sentence summary",
  "line_numbers": [list of affected line numbers],
  "suggested_fix": "corrected code string"
}

Code to analyze:
cursor.execute(f"SELECT * FROM users WHERE id = {user_id}")
"""

# Expected output:
import json
expected_output = {
    "issues": ["SQL injection: user_id is pasted directly into the query"],
    "severity": "high",
    "summary": "Critical SQL injection on line 1.",
    "line_numbers": [1],
    "suggested_fix": "cursor.execute('SELECT * FROM users WHERE id = %s', (user_id,))"
}
print(json.dumps(expected_output, indent=2))

{
  "issues": [
    "SQL injection: user_id is pasted directly into the query"
  ],
  "severity": "high",
  "summary": "Critical SQL injection on line 1.",
  "line_numbers": [
    1
  ],
  "suggested_fix": "cursor.execute('SELECT * FROM users WHERE id = %s', (user_id,))"
}


---
## Session 2: Local vs Cloud Models + Running with Ollama

| Aspect | Local Model | Cloud Model |
|--------|------------|-------------|
| **Privacy** | More secure, data stays local | Data sent to cloud servers |
| **Cost** | Free after hardware | Pay-per-token |
| **Speed** | No network round-trip (hardware-dependent) | Network overhead |
| **Setup** | Ollama | API Key |

### Best Practices for Local Model Prompting
1. **Role + Task + Format** — always give the model a role, specific task, and exact output shape
2. **Positive instructions** — say what TO do ("Output only code") not what to avoid
3. **Few-shot examples** — add 1–2 input/output examples; highest-ROI technique for local models

In [5]:
# ── Minimal Local Assistant Loop ─────────────────────────────────────────────
# Make sure Ollama is running: `ollama serve` in a terminal

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

def create_local_assistant(model_name="gemma2:2b", temperature=0.3):
    """Create a simple local chat assistant using Ollama."""
    llm = ChatOllama(model=model_name, temperature=temperature)
    return llm

def chat(llm, user_input, system_prompt=None):
    """Send a message and get a response."""
    messages = []
    if system_prompt:
        messages.append(SystemMessage(content=system_prompt))
    messages.append(HumanMessage(content=user_input))
    response = llm.invoke(messages)
    return response.content

# --- Run it ---
system = """
You are a senior backend engineer specialised in Python and PostgreSQL.
RULES:
1. Only answer questions about Python, SQL, or system design.
2. Always show code before explaining it.
3. Maximum response length: 200 words.
"""

llm = create_local_assistant()
response = chat(llm, "What is a list comprehension?", system_prompt=system)
print(response)

Certainly! Let's delve into list comprehensions in Python.

**What are List Comprehensions?**

List comprehensions provide a concise way to create lists based on existing iterables (like lists, tuples, or ranges). They combine a `for` loop and an expression within square brackets (`[]`) for efficient list construction. 

**Syntax:**

```python
[expression for item in iterable if condition] 
```

* **expression:** The value you want to assign to each element of the new list.
* **item:** A variable representing each individual `element` from the `iterable`.
* **iterable:**  The sequence (e.g., a list, tuple, range) you're iterating over.
* **condition (optional):** An optional filter that determines which items are included in the new list.

**Example:**

```python
squares = [x**2 for x in range(1, 6)]  # Squares of numbers from 1 to 5
print(squares)  # Output: [1, 4, 9, 16, 25]
```


**Benefits:**

* **Conciseness:**  They replace multiple lines of traditional for loops.
* **Readability

---
## Session 3: Tool Calling + Agent Service

### What makes an "agent"?
An agent combines: **Model** (LLM brain) + **Tools** (real-world actions) + **Prompt** (standing instructions) + **Memory** (persistence across sessions)

### Part A: Standalone Agent Script (`agent_local.py`)

In [7]:
# ── Step 1: Define Custom Tools ──────────────────────────────────────────────
import datetime
from langchain.tools import tool

@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time, formatted according to the
    provided Python strftime format string.
    Use this tool whenever the user asks for the current date, time, or both.
    Example format strings: '%Y-%m-%d' for date, '%H:%M:%S' for time.
    If no format is specified, defaults to '%Y-%m-%d %H:%M:%S'.
    """
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

# List of tools the agent can use
tools = [get_current_datetime]

print("✅ Custom tool defined.")
print("Tool name:", get_current_datetime.name)
print("Tool description:", get_current_datetime.description)

✅ Custom tool defined.
Tool name: get_current_datetime
Tool description: Returns the current date and time, formatted according to the
provided Python strftime format string.
Use this tool whenever the user asks for the current date, time, or both.
Example format strings: '%Y-%m-%d' for date, '%H:%M:%S' for time.
If no format is specified, defaults to '%Y-%m-%d %H:%M:%S'.


In [8]:
# ── Step 2: Set Up the Agent's LLM ───────────────────────────────────────────
from langchain_ollama import ChatOllama

def get_agent_llm(model_name="qwen3:8b", temperature=0):
    """Initializes the ChatOllama model for the agent."""
    # Ensure Ollama server is running: ollama serve
    llm = ChatOllama(
        model=model_name,
        temperature=temperature  # Lower temperature for predictable tool use
        # num_ctx=8192  # Uncomment for longer conversations
    )
    print(f"✅ Initialized ChatOllama agent LLM with model: {model_name}")
    return llm

# agent_llm = get_agent_llm()  # Uncomment to initialize

In [15]:
# ── Step 3: Create the Agent Prompt ──────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def get_agent_prompt():
    """Creates a ReAct-style prompt compatible with Ollama/local models."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant. You have access to the following tools:

{tools}

To use a tool, respond with:
Action: <tool_name>
Action Input: <input>

After getting the result, continue reasoning until you have a final answer.
Always end with:
Final Answer: <your answer>"""),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    print("✅ Agent prompt created (local, no Hub needed)")
    return prompt

agent_prompt = get_agent_prompt()

✅ Agent prompt created (local, no Hub needed)


In [ ]:
# Step 4
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def build_agent(llm, tools):
    """Builds a ReAct agent compatible with local Ollama models."""
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant. You have access to the following tools: {tools}

Use this format:
Thought: think about what to do
Action: tool name
Action Input: input to the tool
Observation: result
... repeat if needed ...
Final Answer: your final answer"""),
        MessagesPlaceholder(variable_name="messages"),
    ])

    agent = create_react_agent(llm, tools)
    print("✅ ReAct Agent created.")
    return agent

def run_agent(agent, user_input):
    """Runs the agent with the given input."""
    print(f"\nInvoking agent...")
    print(f"Input: {user_input}")
    
    response = agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    
    output = response["messages"][-1].content
    print("\nAgent Response:")
    print(output)
    return output

print("✅ Agent builder functions defined.")

✅ Agent builder functions defined.


In [23]:
# ── Run the Full Agent Pipeline ───────────────────────────────────────────────
# Make sure Ollama is running (it should already be running in background)

# 1. Get LLM
agent_llm = get_agent_llm(model_name="qwen3:8b")

# 2. Build Agent (prompt is built inside build_agent now)
agent = build_agent(agent_llm, tools)

# 3. Run queries
run_agent(agent, "What is the current date?")
run_agent(agent, "What time is it right now? Use HH:MM format.")
run_agent(agent, "Tell me a joke.")  # Should NOT use any tool

✅ Initialized ChatOllama agent LLM with model: qwen3:8b
✅ ReAct Agent created.

Invoking agent...
Input: What is the current date?


C:\Users\ALPHV\AppData\Local\Temp\ipykernel_1336\3102487812.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)



Agent Response:
The current date is **2026-04-07**. Let me know if you need further assistance!

Invoking agent...
Input: What time is it right now? Use HH:MM format.

Agent Response:
The current time is **16:35**.

Invoking agent...
Input: Tell me a joke.

Agent Response:
Here's a classic joke for you:

Why don't scientists trust atoms?
Because they make up everything! 😄

Let me know if you'd like another!


"Here's a classic joke for you:\n\nWhy don't scientists trust atoms?\nBecause they make up everything! 😄\n\nLet me know if you'd like another!"

---
### Part B: FastAPI Agent API

> **Note:** FastAPI apps can't run directly inside Jupyter. Save the cell below as `agent_api.py` and run:
> ```bash
> uvicorn agent_api:app --reload
> ```
> Then visit: http://localhost:8000/docs

In [ ]:
"""
FastAPI Agent API with LangChain Tool Calling + Ollama
------------------------------------------------------
Run:
  1. ollama serve
  2. ollama pull qwen2.5:1.5b
  3. pip install fastapi uvicorn langchain langchain-ollama langgraph python-dotenv
  4. uvicorn agent_api:app --reload
  5. Open http://localhost:8000/docs
"""

import datetime
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# ── Pydantic Schemas ──────────────────────────────────────────────────────────
class AgentRequest(BaseModel):
    query: str = Field(..., min_length=1, examples=["What is the current date?"])
    model: str = Field(default="qwen2.5:1.5b", examples=["qwen2.5:1.5b", "llama3.2:1b"])
    temperature: float = Field(default=0.0, ge=0.0, le=1.0)

class ToolCall(BaseModel):
    tool_name: str
    tool_output: str

class AgentResponse(BaseModel):
    query: str
    answer: str
    model: str
    tools_used: list[ToolCall] = []

# ── Custom Tools ──────────────────────────────────────────────────────────────
@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time.
    Use this tool whenever the user asks for the current date, time, or both.
    """
    try:
        return datetime.datetime.now().strftime(format)
    except Exception as e:
        return f"Error formatting date/time: {e}"

@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool when the user asks to calculate something.
    Only supports basic arithmetic: +, -, *, /, ().
    """
    allowed = set("0123456789+-*/.() ")
    if not all(ch in allowed for ch in expression):
        return "Error: expression contains invalid characters."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error evaluating expression: {e}"

TOOLS = [get_current_datetime, calculator]

# ── Agent Factory ─────────────────────────────────────────────────────────────
def build_agent(model_name: str, temperature: float):
    """Build a LangGraph ReAct agent for a given model and temperature."""
    llm = ChatOllama(model=model_name, temperature=temperature)
    agent = create_react_agent(llm, TOOLS)
    return agent

# ── FastAPI App ───────────────────────────────────────────────────────────────
default_agent = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global default_agent
    print("⏳ Building default agent (qwen2.5:1.5b) ...")
    default_agent = build_agent("qwen2.5:1.5b", temperature=0.0)
    print("✅ Agent ready.")
    yield
    print("🛑 Shutting down.")

app = FastAPI(
    title="LangChain Agent API",
    description="FastAPI wrapper around a LangGraph ReAct agent powered by Ollama.",
    version="2.0.0",
    lifespan=lifespan,
)

# ── Routes ────────────────────────────────────────────────────────────────────
@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/agent/chat", response_model=AgentResponse)
def agent_chat(req: AgentRequest):
    """Send a query to the agent and get a response with tool call logs."""
    
    if req.model == "qwen2.5:1.5b" and req.temperature == 0.0 and default_agent:
        agent = default_agent
    else:
        agent = build_agent(req.model, req.temperature)

    try:
        result = agent.invoke({
            "messages": [HumanMessage(content=req.query)]
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Agent error: {e}")

    # Extract tool calls from message history
    tools_used = []
    for msg in result["messages"]:
        if hasattr(msg, "name") and msg.name in [t.name for t in TOOLS]:
            tools_used.append(ToolCall(
                tool_name=msg.name,
                tool_output=str(msg.content)
            ))

    answer = result["messages"][-1].content

    return AgentResponse(
        query=req.query,
        answer=answer,
        model=req.model,
        tools_used=tools_used,
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("agent_api:app", host="0.0.0.0", port=8000, reload=True)


# print("✅ agent_api.py saved! Run it with:")
# print("   uvicorn agent_api:app --reload")

---
# 📅 DAY 2

## Session 4: RAG Grounding with LangChain + ChromaDB

### RAG Fundamentals
| Concept | What It Does |
|---------|-------------|
| **Chunking** | Breaks large docs into smaller pieces for the LLM |
| **Embedding** | Converts text to vectors; similar meanings → close vectors |
| **Retrieval** | Finds top-k chunks most similar to user's query |
| **Citation** | Each chunk carries metadata (source, page) for verifiability |

### Setup: Put your PDF in a `data/` folder
```
your_project/
├── data/
│   └── mydocument.pdf   ← your file here
├── chroma_db/           ← auto-created by ChromaDB
└── rag_local.py
```

In [26]:
# ── Step 1: Load Documents ────────────────────────────────────────────────────
import os
from langchain_community.document_loaders import PyPDFLoader

DATA_PATH = "documents/"
PDF_FILENAME = "sample_journal.pdf"  # ← Replace with your PDF filename

def load_documents():
    """Loads documents from the specified data path."""
    pdf_path = os.path.join(DATA_PATH, PDF_FILENAME)
    loader = PyPDFLoader(pdf_path)
    # Alternative: UnstructuredPDFLoader(pdf_path)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} page(s) from {pdf_path}")
    return documents

documents = load_documents()  # Uncomment after placing your PDF in data/

✅ Loaded 25 page(s) from documents/sample_journal.pdf


In [27]:
# ── Step 2: Split Documents ───────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """Splits documents into smaller chunks suitable for embedding."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,       # Max characters per chunk
        chunk_overlap=200,     # Overlap between chunks to maintain context
        length_function=len,
        is_separator_regex=False,
    )
    all_splits = text_splitter.split_documents(documents)
    print(f"✅ Split into {len(all_splits)} chunks")
    return all_splits

chunks = split_documents(documents)  # Uncomment after loading documents

✅ Split into 114 chunks


In [28]:
# ── Step 3: Configure Embedding Model ─────────────────────────────────────────
# First pull the model in terminal: ollama pull nomic-embed-text

from langchain_ollama import OllamaEmbeddings

def get_embedding_function(model_name="nomic-embed-text"):
    """Initializes the Ollama embedding function."""
    # Ensure Ollama server is running: ollama serve
    embeddings = OllamaEmbeddings(model=model_name)
    print(f"✅ Initialized Ollama embeddings with model: {model_name}")
    return embeddings

embedding_function = get_embedding_function()  # Uncomment to initialize

✅ Initialized Ollama embeddings with model: nomic-embed-text


In [29]:
# ── Step 4: Set Up ChromaDB Vector Store ──────────────────────────────────────
from langchain_community.vectorstores import Chroma

CHROMA_PATH = "chroma_db"  # Directory to store ChromaDB data

def get_vector_store(embedding_function, persist_directory=CHROMA_PATH):
    """Initializes or loads the Chroma vector store."""
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_function
    )
    print(f"✅ Vector store initialized/loaded from: {persist_directory}")
    return vectorstore

def index_documents(chunks, embedding_function, persist_directory=CHROMA_PATH):
    """Indexes document chunks into the Chroma vector store."""
    print(f"Indexing {len(chunks)} chunks...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory=persist_directory
    )
    vectorstore.persist()  # Save to disk
    print(f"✅ Indexing complete. Data saved to: {persist_directory}")
    return vectorstore

# Load existing DB (after initial indexing):
embedding_function = get_embedding_function()
vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

print("✅ Vector store functions defined.")

✅ Initialized Ollama embeddings with model: nomic-embed-text


C:\Users\ALPHV\AppData\Local\Temp\ipykernel_1336\617273936.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)


✅ Vector store functions defined.


In [30]:
# ── Step 5: Build the RAG Chain ───────────────────────────────────────────────
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(vector_store, llm_model_name="qwen2.5:1.5b", context_window=8192):
    """Creates the RAG chain using LangChain Expression Language (LCEL)."""

    # Initialize the LLM
    llm = ChatOllama(
        model=llm_model_name,
        temperature=0,           # Lower temp = more factual RAG answers
        num_ctx=context_window   # IMPORTANT: set context window
    )
    print(f"✅ Initialized ChatOllama with model: {llm_model_name}, ctx: {context_window}")

    # Create the retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",  # Or "mmr" for diversity
        search_kwargs={'k': 3}     # Retrieve top 3 relevant chunks
    )
    print("✅ Retriever initialized.")

    # Define the prompt template
    template = """Answer the question based ONLY on the following context:

{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)
    print("✅ Prompt template created.")

    # Build the LCEL RAG chain: retrieve → prompt → LLM → parse
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("✅ RAG chain created.")
    return rag_chain

# ── Step 6: Query Your Documents ─────────────────────────────────────────────
def query_rag(chain, question):
    """Queries the RAG chain and prints the response."""
    print(f"\nQuestion: {question}")
    response = chain.invoke(question)
    print("\nResponse:")
    print(response)
    return response

print("✅ RAG chain functions defined.")

✅ RAG chain functions defined.


In [31]:
# ── Full RAG Pipeline (run after placing your PDF in data/) ──────────────────
# Make sure: ollama serve is running + nomic-embed-text is pulled

# 1. Load Documents
docs = load_documents()

# 2. Split
chunks = split_documents(docs)

# 3. Embed
embedding_function = get_embedding_function()

# 4. Index (only needed once — comment out after first run)
vector_store = index_documents(chunks, embedding_function)
# To reload an existing index instead:
# vector_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

# 5. Build RAG Chain
rag_chain = create_rag_chain(vector_store, llm_model_name="qwen2.5:1.5b")

# 6. Query
query_rag(rag_chain, "What is the main topic of the document?")
query_rag(rag_chain, "Summarize the introduction section.")

✅ Loaded 25 page(s) from documents/sample_journal.pdf
✅ Split into 114 chunks
✅ Initialized Ollama embeddings with model: nomic-embed-text
Indexing 114 chunks...


C:\Users\ALPHV\AppData\Local\Temp\ipykernel_1336\617273936.py:23: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()  # Save to disk


✅ Indexing complete. Data saved to: chroma_db
✅ Initialized ChatOllama with model: qwen3:8b, ctx: 8192
✅ Retriever initialized.
✅ Prompt template created.
✅ RAG chain created.

Question: What is the main topic of the document?

Response:
The main topic of the document is the **AMFormer-based framework for accident responsibility attribution**, which focuses on using interpretable machine learning analysis to determine responsibility in traffic accidents. The document explores how different traffic accident features (e.g., blood alcohol content, cause identification, and human factors like driving experience) influence responsibility classifications (full, indeterminate, or primary). It emphasizes the framework's ability to provide transparent insights into accident causation and liability assignment.

Question: Summarize the introduction section.

Response:
The introduction section of the document outlines the development of an **AMFormer-based framework** for accident responsibility a

'The introduction section of the document outlines the development of an **AMFormer-based framework** for accident responsibility attribution, emphasizing **interpretable analysis** using traffic accident features. It highlights the importance of accurately assigning responsibility in traffic accidents, addressing limitations in existing methods. The framework integrates traffic-specific features to enhance transparency and reliability in determining liability, supported by references to prior studies on accident risk factors, behavioral variables, and safety implications. The work aims to improve decision-making in accident analysis through a structured, interpretable approach. (Note: The exact introduction text is not explicitly provided in the given context, so this summary is inferred from the title, references, and thematic focus.)'

---
## Session 5: Orchestration Patterns

| Pattern | Control Flow | Best For |
|---------|-------------|----------|
| **Chain** | Linear, deterministic | ETL, Summarize |
| **Router** | Conditional branching | Multi-domain assistants |
| **Planner** | Dynamic branching | Complex research tasks |
| **Tool Loop** | Interactive, self-directed | Agentic tool-use tasks |

### Memory: Session State vs Durable Memory

**Rule:** *Does this need to survive beyond this conversation?*
- Yes → persist it (vector store / DB)
- No → keep it ephemeral in context

In [ ]:
# ── Multi-Step Workflow: Retrieve → Decide → Act → Format ────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:1.5b", temperature=0)

# Step 1 - Retrieve (reuse rag_chain retriever from Session 4)
# Step 2 - Decide: classify the query intent
classify_prompt = ChatPromptTemplate.from_template("""
Given this user question, classify the intent as one of:
[summarize, compare, explain, other]

Question: {question}
Intent (one word only):
""")

classify_chain = classify_prompt | llm | StrOutputParser()

# Step 3 - Act: route to different response formats based on intent
format_prompt = ChatPromptTemplate.from_template("""
Context: {context}
Question: {question}
Intent: {intent}

Based on the intent, format your answer appropriately:
- summarize: bullet points
- compare: a table
- explain: step-by-step
- other: plain text

Answer:
""")

format_chain = format_prompt | llm | StrOutputParser()

print("✅ Orchestration chain defined.")
print("Use classify_chain.invoke({'question': '...'}) to test classification.")

✅ Orchestration chain defined.
Use classify_chain.invoke({'question': '...'}) to test classification.


In [35]:
classify_chain.invoke({'question': 'Help me rearrange'})

'explain'

---
## Session 6: Tracing + Evaluations with LangSmith

### Setup LangSmith
```bash
pip install langsmith
```

Set environment variables (create a `.env` file):
```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=your_langsmith_api_key
LANGCHAIN_PROJECT=my-rag-project
```

In [ ]:
# ── Step 1: Add @traceable to your RAG query function ────────────────────────
# Add this import and decorator to enable automatic tracing in LangSmith

from dotenv import load_dotenv
load_dotenv()  # Loads LANGCHAIN_TRACING_V2 and LANGCHAIN_API_KEY from .env

from langsmith import traceable

@traceable(name="rag-query")
def query_rag_traced(chain, question):
    """Queries the RAG chain and prints the response. Auto-traced in LangSmith."""
    print(f"\nQuerying RAG chain...")
    print(f"Question: {question}")
    response = chain.invoke(question)
    print("\nResponse:")
    print(response)
    return response  # IMPORTANT: return so LangSmith captures the output

print("✅ Traceable query function defined.")

In [ ]:
# ── Step 2: Log Feedback After Each Query ─────────────────────────────────────
import langsmith as ls
from langsmith import Client

client = Client()

def query_rag_with_feedback(chain, question, expected_answer=None):
    """Queries and optionally logs correctness feedback to LangSmith."""
    with ls.trace("rag-query", inputs={"question": question}) as run:
        response = chain.invoke(question)
        run.end(outputs={"answer": response})

        # Log pass/fail if you have a reference answer
        if expected_answer:
            score = 1.0 if expected_answer.lower() in response.lower() else 0.0
            client.create_feedback(
                run_id=run.id,
                key="correctness",
                score=score,
                comment=f"Expected: {expected_answer}"
            )
    return response

print("✅ Feedback logging function defined.")

In [ ]:
# ── Step 3: Run Evaluations Against a Golden Set ──────────────────────────────
from langsmith.evaluation import evaluate, LangChainStringEvaluator

def run_evals(vector_store):
    """Run LangSmith evals against a golden Q&A set."""
    client = Client()
    dataset_name = "local-rag-golden-set"

    # Build the dataset (only once)
    if not any(d.name == dataset_name for d in client.list_datasets()):
        dataset = client.create_dataset(dataset_name)
        client.create_examples(
            inputs=[
                {"question": "What is the main topic of the document?"},
                {"question": "Summarize the introduction section."},
                # Add more Q&A pairs here ↓
            ],
            outputs=[
                {"answer": "...your expected answer 1..."},
                {"answer": "...your expected answer 2..."},
            ],
            dataset_id=dataset.id
        )

    # Wrap the RAG chain for the evaluator
    rag_chain = create_rag_chain(vector_store, llm_model_name="qwen3:8b")

    def run_rag(inputs):
        return {"output": rag_chain.invoke(inputs["question"])}

    # Run with LLM-as-judge
    results = evaluate(
        run_rag,
        data=dataset_name,
        evaluators=[
            LangChainStringEvaluator(
                "labeled_score_string",
                config={"criteria": "correctness", "normalize_by": 10}
            )
        ],
        experiment_prefix="qwen3-chunk-1000",  # Change per experiment
    )
    print(results)

# run_evals(vector_store)  # Uncomment to run evaluations
print("✅ Eval function defined. Uncomment run_evals(vector_store) to execute.")

---
## 🚀 Final Project: Ship a Web-Ready Agent

**Combine everything into one production-shaped service:**

```
your_project/
├── data/                    ← your PDF documents
├── chroma_db/               ← ChromaDB vector store (auto-created)
├── agent_api.py             ← FastAPI agent + tool calling (Session 3)
├── rag_local.py             ← RAG pipeline (Session 4)
├── .env                     ← API keys & LangSmith config
└── requirements.txt
```

### Checklist
- [ ] FastAPI agent API + tool calling (Session 3)
- [ ] RAG with ChromaDB + document loaders (Session 4)
- [ ] Orchestration chains (Session 5)
- [ ] Tracing + evaluation with LangSmith (Session 6)
- [ ] Runs locally with Ollama

### requirements.txt
```
fastapi
uvicorn
langchain
langchain-ollama
langchain-community
langchain-text-splitters
langchainhub
chromadb
pypdf
langsmith
python-dotenv
pydantic
```